In [0]:
# ============================================================
# UPDATE WATERMARK - UPDATED FOR CI/CD
# ============================================================

import os
from datetime import datetime
from pyspark.sql.functions import *

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    raw_container = 'raw-dev'
elif env == 'TEST':
    raw_container = 'raw-test'
else:  # PROD
    raw_container = 'raw'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
WATERMARK_PATH = f"abfss://{raw_container}@{storage_account_name}.dfs.core.windows.net/watermark"
TEMP_PATH = f"abfss://{raw_container}@{storage_account_name}.dfs.core.windows.net/watermark/temp"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 RAW Container: {raw_container}")
print(f"📍 WATERMARK Path: {WATERMARK_PATH}")
print(f"{'='*55}\n")

# ============================================================
# YOUR EXISTING CODE (NO CHANGES NEEDED BELOW!)
# ============================================================

# current time
current_run_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

df_new_watermark = spark.createDataFrame(
    [(current_run_time,)], ["last_load_date"]
)

# write temp
df_new_watermark.coalesce(1).write\
.mode("overwrite")\
.option("header","true")\
.csv(TEMP_PATH)

# get part file
files = dbutils.fs.ls(TEMP_PATH)

for file in files:
    if file.name.startswith("part"):
        dbutils.fs.mv(
            file.path,
            WATERMARK_PATH+"/watermark.csv"
        )

# cleanup temp
dbutils.fs.rm(TEMP_PATH, True)

print(f"\n{'='*55}")
print(f"✅ Watermark updated for {env} environment")
print(f"📅 Last load date: {current_run_time}")
print(f"{'='*55}")

/root/.ipykernel/11825/command-7813510373351797-2173335009:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_run_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


✅ Watermark updated to: 2026-04-02 05:38:49


In [0]:
dbutils.notebook.exit("SUCCESS")